# UHCL
this is an implementation of UHCL in pytorch. UHCL is a loss function that unifies represntation leaning, anomaly detection and hiearachical contrastive learning. 

## Intro
The set of all possible data-points within a space are called data-space. the following framework assumes that each given input-context (main dataset that we aim to develop an AI application for) is a subset of broader set which is data-space. the data-space is mostly noisy data-points, but there are data-points that are not noise nor in the input-context. this frame work creates a context hiearchy 
Noise --> outlier --> inlier (--> positive-inliers) and try to maintain a certain amount of similarity as we trace down the hiearchy tree.



In [ ]:

import os
import random
from typing import List, Tuple, Dict
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from sklearn.metrics import roc_auc_score
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from collections import defaultdict 


# CONFIG
each experiment parameter is mentioned in this config to easily tinker with parameters, 

In [3]:
config = {
    "experiment_name": "2222", # Changed experiment name to avoid overwriting
    "inlier_dataset": "cifar100",
    "outlier_dataset": "cifar100",
    
    "epochs":100,
    "batch_size": 100,
    "anchors_per_fine": 2,
    "lr": 4e-5,
    "aux_weight": 0.000,
    "weight_decay": 0.000,   
    "alpha": 0.01,              # alpha[0] = alpha
    "alpha_decay": 0.9,         # alpha[j] = alpha * (alpha_decay ** j)
                                # the hierarchical design of UHCL ensures alpha[j] in each hiearchy and 
                                # it doesn't require calculation inside hiearchical loop
    "lambda_pos": 0.0,
    "embed_dim": 2,
    "num_workers": 8,
    "pretrained": True,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "ckpt_dir": "./content/checkpoints",
    "resume_ckpt_path": None, # If specified uses a .pth file to resume trainig
                              # if None it does the training from scracth (pretrained or random initilized)
    
    # "resume_ckpt_path": "./content/checkpoints/sssss3_last.pth",
    
    
    # "mode": "supervised",   #TBA "supervised" or "selfsupervised" 
    # "use_noise": True,      #TBA
    # "balanced": False,      # if True: balanced sampling for MNIST family; TBA
}
os.makedirs(config["ckpt_dir"], exist_ok=True)


def set_seed(seed=39):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

class ToTensorSafe:
    def __call__(self, pic):
        if isinstance(pic, torch.Tensor):
            return pic.float() / 255. if pic.max() > 1 else pic.float()
        return transforms.functional.to_tensor(pic)

#  DATALOADERS


In [4]:

IMG_SIZE = {"mnist": 28, "fashionmnist": 28, "emnist": 28, "cifar10": 32, "cifar100": 32, "svhn": 32}

def get_transforms(name, mode="train"):
    size = IMG_SIZE.get(name, 32)
    tensorizer = ToTensorSafe()
    if name in ["mnist", "fashionmnist", "emnist"]:
        # grayscale datasets: no ImageNet norm
        if mode == "train":
            return transforms.Compose([tensorizer])
        else:
            return transforms.Compose([transforms.Resize(size), tensorizer])

    # color image datasets (CIFAR*)
    if mode == "train":
        return transforms.Compose([
            # transforms.RandomResizedCrop(
            #     size,
            #     scale=(0.80, 1.00),      # light zoom variations
            #     ratio=(0.9, 1.1)         # avoid extreme aspect distortions
            # ),
            # transforms.RandomHorizontalFlip(p=0.5),
            
            # Color augmentations
            transforms.ColorJitter(
                brightness=0.4,
                contrast=0.4,
                saturation=0.4,
                hue=0.1
            ),
            
            transforms.RandomGrayscale(p=0.2),    # 10% chance grayscale
            # transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), # slight blur
            
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)
            ),
        ])
    
    else:
        # Test-time: no augmentation except resizing + normalization
        return transforms.Compose([
            transforms.Resize(size),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)
            ),
        ])
class RandomNoiseDataset(Dataset):
      def __init__(self, num: int, shape: Tuple[int, int, int]):
            self.num = num; self.shape = shape
      def __len__(self): return self.num
      def __getitem__(self, idx): return torch.rand(self.shape), -1

In [5]:
import copy
import random
import numpy as np
import torch
from torch.utils.data import Subset

def get_random_coarse_split(dataset, split_ratio=0.5, seed=None):
    """
    Randomly selects a subset of coarse labels to be 'inliers'.
    Returns a set of inlier coarse labels.
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    all_coarse_labels = list(dataset.coarse_label_indices.keys())
    random.shuffle(all_coarse_labels)
    
    n_inliers = int(len(all_coarse_labels) * split_ratio)
    inlier_labels = set(all_coarse_labels[:n_inliers])
    
    print(f"Split generated: {len(inlier_labels)} Inlier Coarse Classes")
    return inlier_labels

def split_dataset_by_labels(full_dataset, inlier_coarse_labels):
    """
    Splits a dataset into in/out sets based on a PRE-DEFINED set of coarse labels.
    """
    # 1. Identify Outlier Indices for the 'out_dataset'
    #    Iterate through all coarse labels in the dataset; if not in inlier list, it's an outlier.
    outlier_indices = []
    dataset_coarse_keys = full_dataset.coarse_label_indices.keys()
    
    for c_lbl in dataset_coarse_keys:
        if c_lbl not in inlier_coarse_labels:
            outlier_indices.extend(full_dataset.coarse_label_indices[c_lbl])
            
    out_dataset = Subset(full_dataset, outlier_indices)

    # 2. Prepare Inlier Dataset (Deep copy indices to avoid mutation issues)
    in_dataset = copy.copy(full_dataset)
    in_dataset.fine_label_indices = copy.deepcopy(full_dataset.fine_label_indices)
    in_dataset.coarse_label_indices = copy.deepcopy(full_dataset.coarse_label_indices)

    # Filter Coarse Indices: Keep only inlier keys
    in_dataset.coarse_label_indices = {
        c: idxs for c, idxs in in_dataset.coarse_label_indices.items() 
        if c in inlier_coarse_labels
    }

    # Filter Fine Indices: Keep only fine labels belonging to inlier coarse classes
    valid_fine_labels = []
    for fine_lbl, indices in in_dataset.fine_label_indices.items():
        if not indices: continue
        
        # Check coarse label of the first sample in this fine class
        # Access original dataset to get (x, fine, coarse) safely
        _, _, sample_coarse = full_dataset[indices[0]]
        
        if sample_coarse in inlier_coarse_labels:
            valid_fine_labels.append(fine_lbl)
            
    in_dataset.fine_label_indices = {
        f: in_dataset.fine_label_indices[f] for f in valid_fine_labels
    }

    return in_dataset, out_dataset

In [6]:
import torch
from torchvision import datasets
import numpy as np
from collections import defaultdict

class CIFAR100Hierarchy(datasets.CIFAR100):
    """
    Custom CIFAR-100 dataset that exposes both fine and coarse labels.
    CIFAR-100 has 20 coarse categories (superclasses), each containing 5 fine categories.
    Includes indexing for efficient sampling.
    """
    def __init__(self, root, train=True, transform=None, target_transform=None, download=False):
        super().__init__(root, train=train, transform=transform,
                         target_transform=target_transform, download=download)

        # Define the hierarchy: mapping from fine labels to coarse labels
        # Each key is a coarse label, and the value is a list of fine labels in that category
        self.coarse_to_fine = {
            0: [4, 30, 55, 72, 95],   # aquatic mammals
            1: [1, 32, 67, 73, 91],   # fish
            2: [54, 62, 70, 82, 92],  # flowers
            3: [9, 10, 16, 28, 61],   # food containers
            4: [0, 51, 53, 57, 83],   # fruit and vegetables
            5: [22, 39, 40, 86, 87],  # household electrical devices
            6: [5, 20, 25, 84, 94],   # household furniture
            7: [6, 7, 14, 18, 24],    # insects
            8: [3, 42, 43, 88, 97],   # large carnivores
            9: [12, 17, 37, 68, 76],  # large man-made outdoor things
            10: [23, 33, 49, 60, 71], # large natural outdoor scenes
            11: [15, 19, 21, 31, 38], # large omnivores and herbivores
            12: [34, 63, 64, 66, 75], # medium-sized mammals
            13: [26, 45, 77, 79, 99], # non-insect invertebrates
            14: [2, 11, 35, 46, 98],  # people
            15: [27, 29, 44, 78, 93], # reptiles
            16: [36, 50, 65, 74, 80], # small mammals
            17: [47, 52, 56, 59, 96], # trees
            18: [8, 13, 48, 58, 90],  # vehicles 1
            19: [41, 69, 81, 85, 89], # vehicles 2
        }

        # Create a mapping from fine labels to coarse labels
        self.fine_to_coarse = {}
        for coarse_label, fine_labels in self.coarse_to_fine.items():
            for fine_label in fine_labels:
                self.fine_to_coarse[fine_label] = coarse_label

        # Create coarse_targets array for all samples
        self.coarse_targets = [self.fine_to_coarse[fine_label] for fine_label in self.targets]

        # Create indices for efficient sampling
        self.fine_label_indices = defaultdict(list)
        self.coarse_label_indices = defaultdict(list)

        for i, (fine_label, coarse_label) in enumerate(zip(self.targets, self.coarse_targets)):
            self.fine_label_indices[fine_label].append(i)
            self.coarse_label_indices[coarse_label].append(i)


    def __getitem__(self, index):
        """
        Args:
            index (int): Index
        Returns:
            tuple: (image, fine_target, coarse_target) where fine_target is the index of the target class.
        """
        img, fine_target = super().__getitem__(index)
        coarse_target = self.coarse_targets[index]
        return img, fine_target, coarse_target

class CIFAR10Indexed(datasets.CIFAR10):
    """
    Custom CIFAR-10 dataset with pre-computed indices for faster sampling.
    """
    def __init__(self, root, train=True, transform=None, target_transform=None, download=False):
        super().__init__(root, train=train, transform=transform,
                         target_transform=target_transform, download=download)

        # Create indices for efficient sampling
        self.label_indices = defaultdict(list)
        for i, label in enumerate(self.targets):
            self.label_indices[label].append(i)

In [7]:
def sample_hierarchy_batch(in_dataset, out_dataset, noise_dataset,
                           batch_size, device, N = 4, is_hierarchical=False,
                           anchors_per_fine=config["anchors_per_fine"]):

    X_batch = []

    # ALWAYS create label containers
    yfine_batch = []
    ycoarse_batch = []   # will be None for CIFAR10
    y_batch = []         # CIFAR10 fine labels

    if is_hierarchical:
        fine_labels = list(in_dataset.fine_label_indices.keys())
        random.shuffle(fine_labels)

        for fine_label in fine_labels:
            fine_indices = in_dataset.fine_label_indices[fine_label]

            # anchors
            if len(fine_indices) < anchors_per_fine:
                anchors = np.random.choice(fine_indices, anchors_per_fine, replace=True)
            else:
                anchors = np.random.choice(fine_indices, anchors_per_fine, replace=False)

            for idx in anchors:
                try:
                    x0, fine0, coarse0 = in_dataset[idx]

                    # x1: same fine (pos)
                    pos_candidates = [i for i in in_dataset.fine_label_indices[fine0] if i != idx]
                    pos_idx = random.choice(pos_candidates) #if pos_candidates else idx
                    x1, _, _ = in_dataset[pos_idx]

                    # x2: same coarse, diff fine
                    coarse_candidates = [
                        i for i in in_dataset.coarse_label_indices[coarse0]
                        if in_dataset.targets[i] != fine0
                    ]
                    neg_fine_idx = random.choice(coarse_candidates) #if coarse_candidates else idx
                    x2, fine2, _ = in_dataset[neg_fine_idx]

                    # x3: diff coarse
                    all_coarse = list(in_dataset.coarse_label_indices.keys())
                    diff_coarse = [c for c in all_coarse if c != coarse0]
                    rand_coarse = random.choice(diff_coarse)
                    diff_coarse_candidates = in_dataset.coarse_label_indices[rand_coarse]
                    diff_coarse_idx = random.choice(diff_coarse_candidates)
                    x3, fine3, coarse3 = in_dataset[diff_coarse_idx]

                    # x4: outlier
                    # x4, _ = out_dataset[np.random.randint(len(out_dataset))]
                    x4, *rest = out_dataset[np.random.randint(len(out_dataset))]
                    # x5: noise
                    x5, _ = noise_dataset[np.random.randint(len(noise_dataset))]

                    # assemble tuple
                    X_tuple = torch.stack([x0, x1, x2, x3, x4, x5])

                    # labels ONLY for inlier-inlier comparisons (not outlier/not noise)
                    yfine_tuple = [fine0, fine0, fine2, fine3]    # 4 entries
                    ycoarse_tuple = [coarse0, coarse0, coarse0, coarse3]

                    X_batch.append(X_tuple)
                    yfine_batch.append(yfine_tuple)
                    ycoarse_batch.append(ycoarse_tuple)

                except:
                    print ('gotcha')
                    x0, _, _ = in_dataset[idx]
                    fallback = [x0] * N
                    X_batch.append(torch.stack(fallback))

    else:
        # CIFAR10-like
        labels = list(in_dataset.label_indices.keys())
        random.shuffle(labels)

        for label in labels:
            label_indices = in_dataset.label_indices[label]

            if len(label_indices) < anchors_per_fine:
                anchors = np.random.choice(label_indices, anchors_per_fine * 10, replace=True)
            else:
                anchors = np.random.choice(label_indices, anchors_per_fine * 10, replace=False)

            for idx in anchors:
                try:
                    x0, y0 = in_dataset[idx]

                    pos_candidates = [i for i in in_dataset.label_indices[y0] if i != idx]
                    pos_idx = random.choice(pos_candidates) if pos_candidates else idx
                    x1, _ = in_dataset[pos_idx]

                    # diff label
                    diff_labels = [l for l in labels if l != y0]
                    neg_label = random.choice(diff_labels)
                    neg_idx = random.choice(in_dataset.label_indices[neg_label])
                    x2, y2 = in_dataset[neg_idx]

                    x3, _ = out_dataset[np.random.randint(len(out_dataset))]
                    x4, _ = noise_dataset[np.random.randint(len(noise_dataset))]

                    X_tuple = torch.stack([x0, x1, x2, x3, x4])
                    y_tuple = [y0, y0, y2]   # only for inliers

                    X_batch.append(X_tuple)
                    y_batch.append(y_tuple)

                except:
                    x0, _ = in_dataset[idx]
                    fallback = [x0] * N
                    X_batch.append(torch.stack(fallback))

        # convert to tensors
        yfine_batch = y_batch                      # fine = class id
        ycoarse_batch = None                       # not used

    # clip size
    if len(X_batch) > batch_size:
        X_batch = X_batch[:batch_size]
        if yfine_batch is not None:
            yfine_batch = yfine_batch[:batch_size]
        if ycoarse_batch is not None:
            ycoarse_batch = ycoarse_batch[:batch_size]

    X_batch = torch.stack(X_batch).to(device)

    if ycoarse_batch is None:
        ycoarse_batch = torch.zeros((len(yfine_batch), 1), dtype=torch.long)

    return (
        X_batch,
        torch.tensor(yfine_batch, dtype=torch.long, device=device),
        torch.tensor(ycoarse_batch, dtype=torch.long, device=device),
    )


# MODEL & PROJECTION

In [8]:
class SmallResNet(nn.Module):

    def __init__(self, embed_dim=128, pretrained=False, in_channels=3, depth= 18, num_classes=None):
        super().__init__()
        if depth == 50:
            # Use ResNet50 weights if pretrained is True
            net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
            proj_in_features = 2048  # ResNet50 outputs 2048 features before the final layer
        else: # Assuming depth 18 for other cases
            # Use ResNet18 weights if pretrained is True
            net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
            proj_in_features = 512 # ResNet18 outputs 512 features before the final layer

        if in_channels != 3:
            # If the input channels are not 3, we need to modify the first convolutional layer.
            # We keep the original weights for the 3 input channels and initialize the new channels randomly.
            original_conv1 = net.conv1
            net.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
            if pretrained:
                 # Copy weights from the original 3 channels
                 net.conv1.weight.data[:, :3, :, :] = original_conv1.weight.data
                 # Initialize weights for additional channels randomly (e.g., Kaiming initialization)
                 nn.init.kaiming_normal_(net.conv1.weight.data[:, 3:, :, :], mode='fan_out', nonlinearity='relu')


        self.backbone = nn.Sequential(*list(net.children())[:-1])

        # Projector to map backbone features to desired embedding dimension
        self.projector = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), # Add AdaptiveAvgPool2d to ensure 1x1 spatial dimensions
            nn.Flatten(), # Input: [batch_size, features, 1, 1], Output: [batch_size, features] (features = proj_in_features)
            nn.Linear(proj_in_features, 512), # Input: [batch_size, proj_in_features], Output: [batch_size, 512]
            nn.ReLU(inplace=True),            # Input: [batch_size, 512], Output: [batch_size, 512]
            nn.Linear(512, embed_dim),        # Input: [batch_size, 512], Output: [batch_size, embed_dim]
        )

        # Optional classification head for anomaly/noise detection
        self.clf_head = None
        if num_classes is not None:
            self.clf_head = nn.Linear(embed_dim, num_classes)

    def forward_features_and_logits(self, x):
        """
        Returns:
          emb:  [B, D]
          logits: [B, num_classes]   <- from clf_head (inlier/outlier/noise)
        """
        emb = self(x, return_embeddings=True)
    
        if self.clf_head is None:
            raise ValueError("clf_head is not defined. Make sure num_classes was set.")
    
        logits = self.clf_head(emb)
        return emb, logits

    
    #**** there might be a problem with last layer of my model, 
    #     double check if not including N outputs (classification probablities) in the output cause any error

    
    

    def forward(self, x, return_embeddings=True):
        z = self.backbone(x)
        z = self.projector(z)
        emb = F.normalize(z, p=2, dim=1)

        return emb, z
        

# UHCL

In [9]:
def uhcl(emb, N=5, alpha=0.95, alpha_decay=0.8, device=None):
    """
    Averaged-per-partial UHCL loss (first version modified to match the second).
    emb: [B, N, D] — normalized embeddings.
    """

    B, total_levels, D = emb.shape
    S = torch.bmm(emb, emb.transpose(1, 2))  # [B, N, N]

    Total_loss = torch.tensor(0.0, device=device)

    # Identity loss averaged like the second implementation
    # identity_loss = F.relu(S[:, 0, 1] - alpha).mean()

    # Start from level 1 ⇒ positive pairs (i < j)
    total_margin_loss = torch.tensor(0.0, device=device)
    total_margin_loss += F.relu(S[:,0,total_levels-1]).mean()
    alpha_j_base = alpha #* alpha_decay
    if total_levels == 6:
        # margin = [15/16 , 7/8 , 3/4 , 1/2 ]
        margin = [15/16, 7/8 , 3/4 , 1/2]
        
        # print('you are here')
    else:
        margin = [7/8 , 3/4 , 1/2 ]

    for j in range(1 , total_levels - 2):
        # print (j+1)
        margin_loss = F.relu((margin[j-1] - S[:,0,j]))
        total_margin_loss += margin_loss.mean()
    
    for j in range(1, total_levels - 1):
        alpha_j = (1 - margin[j-1])*0.95
        hierarchy_j_loss = torch.tensor(0.0, device=device)
        # alpha_j = alpha_j_base  # can depend on j if you want

        for i in range(0, j):

            partial_loss = torch.tensor(0.0, device=device)

            # Number of negative pairs = j - i + 1
            for k in range(i, j + 1):

                positive_sim = S[:, i, j]                # [B]
                negative_sim = S[:, k, j + 1]           # [B]

                hinge_loss = F.relu(negative_sim - positive_sim + alpha_j)#+ alpha_j)

                # **Average over batch here (like the second function)**
                partial_loss += hinge_loss.mean()

            # Normalize like before
            if j != 1:
                lambda_ij = 1 / (j - i + 1)
            else:
                lambda_ij = 2 / (j - i + 1)

            hierarchy_j_loss += partial_loss * lambda_ij

        # Normalize for number of positive terms (i < j)
        lambda_j = 1 / j
        Total_loss += hierarchy_j_loss * lambda_j

    return Total_loss + total_margin_loss#+ identity_loss


In [10]:
# def uhcl(emb, N=5, alpha=0.95, alpha_decay = 0.8, device=None):
#     """
#     emb: [B, N, D] tensor / embeddings are already normalized
#     calculates pairwise similarity matrix : Batch * Batch^-1 (uses bottom half)
#     considers each i,j pair in the matrix as a positive pair and ensures:
#         for each positive pair i , j:
#         --> S[i,j] > s[k , j+1] ;   i <= K <= j+1
#         --> averages over total loss
#     """
#     B, total_levels, D = emb.shape
#     # alpha = [0.94 , 0.75 , 0.65, 0.5 , 0.3]
#     # Compute pairwise similarity matrix
#     S = torch.bmm(emb, emb.transpose(1, 2))  # [B, N, N]
    
#     total_loss = torch.zeros(B, device=device)
#     # positive_loss = torch.zeros(B, device=device)
#     identity_loss =  F.relu(S[:, 0, 1] - alpha) # * 1 or s[:,0,0]
#     # For each hierarchy level j (positive level)
#     # Range should start from j other wise we use z[0].z[0] = s[0,0]
#     alpha_j= alpha * alpha_decay
#     for j in range(1 , total_levels - 1): 
#         # For each positive level i (where i < j to avoid self-comparison)
#         hierarchy_j_loss = torch.zeros(B, device=device) # [B]
        
        
#         for i in range(0, j):  
#             positive_sim = S[:, i, j]  # [B]
#             # Negative pairs for a positive pair i,j 
#             # is from next hierarchy level (j+1)
#             pair_ij_loss = torch.zeros(B, device=device) # [B]
#             for k in range(i, j + 1):  
#                 negative_sim = S[:, k, j + 1]  # [B]
                
#                 # Hinge loss
#                 hinge_loss = F.relu(negative_sim - positive_sim*alpha_j)  # [B]                
#                 pair_ij_loss += hinge_loss

#             # lambda_ij is the normalizing coeficient over number of negative pairs for
#             # positive pair i,j
#             # if j == 1:
#             #     lambda_ij = 10 / (j - i + 1)
#             # elif j == 2:
#             #     lambda_ij = 5 / (j - i + 1)
#             # else:
#             lambda_ij = 1 / (j - i + 1)

#             hierarchy_j_loss += pair_ij_loss * lambda_ij

#         # lambda_j is normalizing coeficient over number of positive terms in each hierarchy j
#         lambda_j = 1 / j 
#         total_loss += (hierarchy_j_loss * lambda_j) #+ (positive_loss * 0.08)
    
#     return (total_loss + identity_loss).mean()  # Final average over batch

In [11]:
# def uhcl(emb: List[torch.Tensor], N = 4 , alpha: float=0.15, device=None):
#     """
#     Generalized UHCL loss: levels is a list of embeddings [B x D] for hierarchy levels H1..HN.
#     Enforces that similarity at level i is greater than similarities at all levels j>i.
#     """
#     B, N, D = emb.shape
#     Total_loss = torch.tensor(0.0, device=device)
#     # for i in range(1,N - 1):
#     #     lmbd = (N - i) / sum(range(2, i + 2))   # weighted by N - i becuase of importance
#     for j in range(1 , N - 1):                                            # and normalized on the number of existing terms.
#         hierarchy_loss = torch.tensor(0.0, device=device)
#         lmbd3 = 1 #((N - j) / (j + 1))
#         alpha = 0.15
#         # alpha = (j * 0.08)+ 0.15

#         for i in range(0, j):
#             partial_loss = torch.tensor(0.0, device=device)
#             for k in range(i,j+1):
#                 sim_p = (emb[:, j] * emb[:, i]).sum(dim=1)   # indicates the positives term in each hierarchy
#                 sim_n = (emb[:,j+1] * emb[:,k]).sum(dim=1)   # indicates the negative term
#                 # lmbd1 =1 / (j - k)
#                 lmbd1 = 1
#                 partial_loss += F.relu((sim_n - sim_p + alpha) * lmbd1).mean()
#                 # print (alpha)
#             # lmbd2 = 1 / ((j + 1) - i)
#             lmbd2 = 1
#             hierarchy_loss += partial_loss * lmbd2
#             #lmbd = 1 / (len(range(j,i))+1)
#         Total_loss += hierarchy_loss * lmbd3
#     return Total_loss

In [12]:
# def hyperbolic_margin_loss(x_anchor, x_pos, x_neg, margin=0.5):
#     # x_* are points in Poincare ball
#     d_pos = manifold.dist(x_anchor, x_pos)
#     d_neg = manifold.dist(x_anchor, x_neg)
#     loss = torch.relu(d_pos - d_neg + margin).mean()
#     return loss


# EVAL

In [13]:
def train_clf(model, in_train, out_train, noise_train, device, epochs=10, batch_size=256, lr=1e-3):
    """
    Trains the 3-class classification head (Inlier=0, Outlier=1, Noise=2)
    using frozen embeddings from the trained backbone.
    """
    print("\n--- Training 3-Class Classification Head (Linear Probe) ---")
    
    # 1. Set backbone to eval (freeze it), but we will manually update the head
    model.eval() 
    if model.clf_head is None:
        print("Error: Classification head not initialized.")
        return

    # 2. Extract Features (Frozen Backbone)
    print("Extracting features from frozen backbone...")
    
    @torch.no_grad()
    def get_feats_labels(dataset, label_val):
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4)
        feats = []
        targets = []
        for batch in tqdm(loader, desc=f"Class {label_val}"):
            # Handle tuple unpacking depending on dataset type
            if isinstance(batch, (list, tuple)):
                x = batch[0] # Image is always first
            else:
                x = batch
                
            x = x.to(device)
            # Get normalized embeddings
            _, logits  = model(x)
            feats.append(logits.cpu())
            targets.append(torch.full((logits.size(0),), label_val))
        
        return torch.cat(feats), torch.cat(targets)

    # Extract for all 3 categories
    # Inliers = Class 0
    in_feats, in_labels = get_feats_labels(in_train, 0)
    # Outliers = Class 1 (limit size to match inliers to avoid massive imbalance)
    out_feats, out_labels = get_feats_labels(out_train, 1)
    # Noise = Class 2
    noise_feats, noise_labels = get_feats_labels(noise_train, 2)

    # Balance the training data for the classifier
    min_len = min(len(in_feats), len(out_feats), len(noise_feats))
    # You can choose to use all data, but balancing usually helps convergence for the head
    # Here we just concat everything:
    X_all = torch.cat([in_feats, out_feats, noise_feats])
    y_all = torch.cat([in_labels, out_labels, noise_labels])

    # 3. Create TensorDataset for the Head Training
    head_dataset = torch.utils.data.TensorDataset(X_all, y_all)
    head_loader = DataLoader(head_dataset, batch_size=batch_size, shuffle=True)

    # 4. Setup Optimizer for Head ONLY
    # We are strictly optimizing model.clf_head.parameters()
    head_opt = torch.optim.Adam(model.clf_head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # 5. Train the Head
    model.clf_head.train() # Set head to train mode (for dropout if applicable)
    
    for ep in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        
        for feats, labels in head_loader:
            feats, labels = feats.to(device), labels.to(device)
            
            # Forward pass through HEAD only
            logits = model.clf_head(feats)
            loss = criterion(logits, labels.long())
            
            head_opt.zero_grad()
            loss.backward()
            head_opt.step()
            
            total_loss += loss.item()
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
        avg_loss = total_loss / len(head_loader)
        acc = 100 * correct / total
        print(f"Head Epoch {ep+1}/{epochs} | Loss: {avg_loss:.4f} | Acc: {acc:.2f}%")

    print("Classification Head Training Complete.")

In [14]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
)
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")


# -------------------------------------------------------------------------
#   Extract embeddings
# -------------------------------------------------------------------------
@torch.no_grad()
#Im not sure if I must work with logits or embeddings for this one
#but since it normalizes them at the end I go with logits
def extract_embeddings(dataset, model, device, batch_size=256, hierarchical=False):

    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_features = []
    all_fine_labels = []
    all_coarse_labels = []
    all_classifier_probs = []

    for batch in loader:

        if hierarchical:
            images, fine_labels, coarse_labels = batch
        else:
            images, fine_labels = batch
            coarse_labels = [-1] * len(fine_labels)

        images = images.to(device)

        # Extract embedding (model must support return_embeddings=True)
        _, features_t= model(images) # ****************** change this if anything gone wrong **********************
        features_np = features_t.detach().cpu().numpy()
        all_features.append(features_np)

        all_fine_labels.extend([int(l) for l in fine_labels])
        all_coarse_labels.extend([int(l) for l in coarse_labels])

        # Optional classifier head
        if getattr(model, "clf_head", None) is not None:
            features_tensor = torch.tensor(features_np, dtype=torch.float32).to(device)
            logits = model.clf_head(features_tensor)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            all_classifier_probs.append(probs)

    all_features = np.vstack(all_features)
    all_fine_labels = np.array(all_fine_labels)
    all_coarse_labels = np.array(all_coarse_labels)

    if len(all_classifier_probs):
        all_classifier_probs = np.vstack(all_classifier_probs)
    else:
        all_classifier_probs = np.empty((len(all_features), 0))

    return all_features, all_fine_labels, all_coarse_labels, all_classifier_probs


# -------------------------------------------------------------------------
#   KNN classifier helper
# -------------------------------------------------------------------------
# def compute_knn(train_features, train_labels, test_features, test_labels, k=40):

#     knn = KNeighborsClassifier(n_neighbors=k)
#     knn.fit(train_features, train_labels)

#     train_predictions = knn.predict(train_features)
#     test_predictions = knn.predict(test_features)

#     def metrics(true, pred):
#         return (
#             (true == pred).mean(),
#             precision_score(true, pred, average="weighted", zero_division=0),
#             recall_score(true, pred, average="weighted", zero_division=0),
#             f1_score(true, pred, average="weighted", zero_division=0),
#         )

#     return train_predictions, test_predictions, metrics(train_labels, train_predictions), metrics(test_labels, test_predictions)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

def compute_knn(train_features, train_labels, test_features, test_labels, k=20):
    # Set metric to 'cosine'
    knn = KNeighborsClassifier(n_neighbors=k, metric='cosine')
    knn.fit(train_features, train_labels)

    train_predictions = knn.predict(train_features)
    test_predictions = knn.predict(test_features)

    def metrics(true, pred):
        return (
            (true == pred).mean(),
            precision_score(true, pred, average="weighted", zero_division=0),
            recall_score(true, pred, average="weighted", zero_division=0),
            f1_score(true, pred, average="weighted", zero_division=0),
        )

    return (
        train_predictions, 
        test_predictions, 
        metrics(train_labels, train_predictions), 
        metrics(test_labels, test_predictions)
    )
# -------------------------------------------------------------------------
#   LARGE fine confusion matrix (FULL, READABLE)
# -------------------------------------------------------------------------
def plot_confusion_matrix_large(true_labels, predicted_labels, title, filename):

    matrix = confusion_matrix(true_labels, predicted_labels)
    num_classes = matrix.shape[0]

    plt.figure(figsize=(28, 24))  # Large readable
    sns.heatmap(
        matrix,
        annot=True,
        fmt='d',
        cmap="Blues",
        cbar=False,
        annot_kws={"size": 7}  # readable numbers
    )
    plt.title(f"{title}  ({num_classes}×{num_classes})", fontsize=20)
    plt.xlabel("Predicted", fontsize=16)
    plt.ylabel("True", fontsize=16)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()
# -------------------------------------------------------------------------
#   LARGE Coarse confusion matrix - 
# -------------------------------------------------------------------------
def plot_confusion_matrix_large_coarse(true_labels, predicted_labels, title, filename):
    matrix = confusion_matrix(true_labels, predicted_labels)
    num_classes = matrix.shape[0]

    plt.figure(figsize=(24, 22))  # slightly smaller than fine, but with bigger text
    sns.heatmap(
        matrix,
        annot=True,
        fmt='d',
        cmap="Greens",
        cbar=False,
        annot_kws={"size": 14}   # BIGGER numbers
    )
    plt.title(f"{title} ({num_classes}×{num_classes})", fontsize=24)
    plt.xlabel("Predicted", fontsize=20)
    plt.ylabel("True", fontsize=20)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()


# -------------------------------------------------------------------------
#   Standard t-SNE for fine/coarse
# -------------------------------------------------------------------------
def plot_tsne(features, labels, title, filename, max_samples=2000):

    if len(features) == 0:
        return

    sample_count = min(len(features), max_samples)
    idx = np.random.choice(len(features), sample_count, replace=False)

    selected_features = features[idx]
    selected_labels = labels[idx]

    tsne_2d = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(selected_features)

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(tsne_2d[:, 0], tsne_2d[:, 1], c=selected_labels, cmap="tab20", s=6)
    plt.colorbar(scatter, label="Label")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()


# -------------------------------------------------------------------------
#   NEW: Inlier / Outlier / Noise t-SNE
# -------------------------------------------------------------------------
def plot_tsne_ood(features, labels_0_in_1_out_2_noise, exp_name, filename, max_samples=4000):

    count = min(len(features), max_samples)
    idx = np.random.choice(len(features), count, replace=False)
    f = features[idx]
    l = labels_0_in_1_out_2_noise[idx]

    tsne_2d = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(f)

    plt.figure(figsize=(6, 6))
    plt.scatter(tsne_2d[l == 0, 0], tsne_2d[l == 0, 1], c="g", s=5, label="Inlier")
    plt.scatter(tsne_2d[l == 1, 0], tsne_2d[l == 1, 1], c="r", s=5, label="Outlier")
    plt.scatter(tsne_2d[l == 2, 0], tsne_2d[l == 2, 1], c="b", s=5, label="Noise")
    plt.legend()
    plt.title(f"t-SNE {exp_name} (Inlier / Outlier / Noise)")
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()


# -------------------------------------------------------------------------
#   MAIN EVALUATION
# -------------------------------------------------------------------------
def evaluate_summary(
    model, 
    inlier_train_set, 
    inlier_test_set, 
    outlier_test_set, 
    noise_test_set,
    device, 
    exp_name,
    batch_size=256,
    k_fine=10,
    k_coarse=10
):

    print(f"Evaluating model: {exp_name}")

    hierarchical = (
        isinstance(inlier_test_set, type(inlier_train_set)) and
        hasattr(inlier_test_set, "__class__") and
        "CIFAR100" in inlier_test_set.__class__.__name__
    )

    # ------------------------------------------------------
    #   Extract embeddings
    # ------------------------------------------------------
    train_features, train_fine_labels, train_coarse_labels, train_probs = extract_embeddings(
        inlier_train_set, model, device, batch_size, hierarchical
    )
    test_features, test_fine_labels, test_coarse_labels, test_probs = extract_embeddings(
        inlier_test_set, model, device, batch_size, hierarchical
    )
    out_features, _, _, out_probs = extract_embeddings(
        outlier_test_set, model, device, batch_size, True
    )
    noise_features, _, _, noise_probs = extract_embeddings(
        noise_test_set, model, device, batch_size, False
    )

    # Normalize for cosine-like KNN
    def l2_normalize(x):
        n = np.linalg.norm(x, axis=1, keepdims=True)
        n[n == 0] = 1
        return x / n

    norm_train = l2_normalize(train_features)
    norm_test = l2_normalize(test_features)

    # ------------------------------------------------------
    #   Fine KNN
    # ------------------------------------------------------
    pred_train_fine, pred_test_fine, fine_train_metrics, fine_test_metrics = compute_knn(
        norm_train, train_fine_labels, norm_test, test_fine_labels, k=k_fine
    )

    # BIG CLEAN CONFUSION MATRICES
    plot_confusion_matrix_large(
        train_fine_labels, pred_train_fine,
        title=f"{exp_name} Fine CM (Train)",
        filename=f"{exp_name}_fine_cm_train.png"
    )
    plot_confusion_matrix_large(
        test_fine_labels, pred_test_fine,
        title=f"{exp_name} Fine CM (Test)",
        filename=f"{exp_name}_fine_cm_test.png"
    )

    # ------------------------------------------------------
    #   Coarse KNN
    # ------------------------------------------------------
    if hierarchical:
        mask_train = train_coarse_labels >= 0
        mask_test = test_coarse_labels >= 0

        train_coarse_feats = norm_train[mask_train]
        test_coarse_feats = norm_test[mask_test]
        train_coarse_lbls = train_coarse_labels[mask_train]
        test_coarse_lbls = test_coarse_labels[mask_test]

        if len(train_coarse_lbls) > 0:
            pred_train_coarse, pred_test_coarse, coarse_train_metrics, coarse_test_metrics = compute_knn(
                train_coarse_feats, train_coarse_lbls,
                test_coarse_feats, test_coarse_lbls,
                k=k_coarse
            )

            plot_confusion_matrix_large_coarse(
                train_coarse_lbls, pred_train_coarse,
                title=f"{exp_name} Coarse CM (Train)",
                filename=f"{exp_name}_coarse_cm_train.png"
            )
            plot_confusion_matrix_large_coarse(
                test_coarse_lbls, pred_test_coarse,
                title=f"{exp_name} Coarse CM (Test)",
                filename=f"{exp_name}_coarse_cm_test.png"
            )

        else:
            coarse_train_metrics = coarse_test_metrics = (np.nan,) * 4
    else:
        coarse_train_metrics = coarse_test_metrics = (np.nan,) * 4

    # ------------------------------------------------------
    #   t-SNE visualizations
    # ------------------------------------------------------
    plot_tsne(train_features, train_fine_labels, f"{exp_name} Fine t-SNE (Train)", f"{exp_name}_tsne_fine_train.png")
    plot_tsne(test_features, test_fine_labels, f"{exp_name} Fine t-SNE (Test)", f"{exp_name}_tsne_fine_test.png")

    if hierarchical:
        plot_tsne(train_features, train_coarse_labels, f"{exp_name} Coarse t-SNE (Train)", f"{exp_name}_tsne_coarse_train.png")
        plot_tsne(test_features, test_coarse_labels, f"{exp_name} Coarse t-SNE (Test)", f"{exp_name}_tsne_coarse_test.png")

    # t-SNE of OOD structure (0=inlier, 1=outlier, 2=noise)
    all_features = np.concatenate([test_features, out_features, noise_features], axis=0)
    all_markers = np.concatenate([
        np.zeros(len(test_features)),
        np.ones(len(out_features)),
        np.full(len(noise_features), 2)
    ])
    plot_tsne_ood(all_features, all_markers, exp_name, f"{exp_name}_tsne_inlier_outlier_noise.png")

    # ------------------------------------------------------
    #   AUROC (classifier head)
    # ------------------------------------------------------
    if getattr(model, "clf_head", None) is None:
        return (
            *fine_train_metrics,
            *coarse_train_metrics,
            *fine_test_metrics,
            *coarse_test_metrics,
            np.nan, np.nan, np.nan, np.nan,
            np.nan, np.nan, np.nan, np.nan
        )

    # OOD AUROC
    outlier_scores_in = test_probs[:, 1] if test_probs.shape[1] > 1 else np.zeros(len(test_probs))
    outlier_scores_out = out_probs[:, 1] if out_probs.shape[1] > 1 else np.zeros(len(out_probs))

    noise_scores_in = test_probs[:, 2] if test_probs.shape[1] > 2 else np.zeros(len(test_probs))
    noise_scores_noise = noise_probs[:, 2] if noise_probs.shape[1] > 2 else np.zeros(len(noise_probs))

    # Compute AUROCs safely
    def compute_auroc(scores_a, scores_b):
        y_true = np.concatenate([np.zeros_like(scores_a), np.ones_like(scores_b)])
        y_score = np.concatenate([scores_a, scores_b])
        if len(np.unique(y_true)) == 1:
            return np.nan, np.nan, np.nan, np.nan
        auroc = roc_auc_score(y_true, y_score)
        preds = (y_score > 0.5).astype(int)
        return (
            auroc,
            precision_score(y_true, preds, zero_division=0),
            recall_score(y_true, preds, zero_division=0),
            f1_score(y_true, preds, zero_division=0),
        )

    ood_metrics = compute_auroc(outlier_scores_in, outlier_scores_out)
    noise_metrics = compute_auroc(noise_scores_in, noise_scores_noise)

    # Return all metrics in same structure as before
    return (
        *fine_train_metrics,
        *coarse_train_metrics,
        *fine_test_metrics,
        *coarse_test_metrics,
        *ood_metrics,
        *noise_metrics
    )


In [15]:
# # ============================================================
# #   Mahalanobis & Energy OOD Scoring – Ready-to-run Cell
# # ============================================================

import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve
import scipy.special

EPS = 1e-6
# device = "cuda" if torch.cuda.is_available() else "cpu"


# # ------------------------------------------------------------
# # 1) Extract features & logits from a dataloader
# # ------------------------------------------------------------
# @torch.no_grad()
# def extract_features_and_logits(model, loader, hierarchical = False):
#     model.eval()
#     feats, logits, fine_labels, coarse_labels = [], [], [], []
#     if not hierarchical:
#         for x, y in loader:
#             x = x.to(device)
#             f, lg = model.forward_features_and_logits(x)
#             feats.append(f.cpu())
#             logits.append(lg.cpu())
#             fine_labels.append(y)
#     else:
#         for x, yf , yc in loader:
#             x = x.to(device)
#             f, lg = model.forward_features_and_logits(x)
#             feats.append(f.cpu())
#             logits.append(lg.cpu())
#             fine_labels.append(yf)
#             coarse_labels.append(yc)
#     fine_labels = torch.cat(fine_labels, dim=0)      # [N]
#     if hierarchical:
#         coarse_labels = torch.cat(coarse_labels, dim=0)      # [N]

#     feats = torch.cat(feats, dim=0)        # [N, D]
#     logits = torch.cat(logits, dim=0)      # [N, C]

#     return feats, logits, fine_labels, coarse_labels


# # ------------------------------------------------------------
# # 2) Compute train statistics for Mahalanobis
# # ------------------------------------------------------------
# def compute_maha_stats(model, train_loader, hierarchical):
#     feats, logits, f_labels, c_labels  = extract_features_and_logits(model, train_loader, hierarchical)
#     feats_np = feats.numpy()
#     f_labels_np = f_labels.numpy()
#     c_labels_np = c_labels.numpy()
#     num_fine_classes = int(f_labels_np.max()) + 1
#     num_coarse_classes = int(c_labels_np.max()) + 1

#     D = feats_np.shape[1]

#     # class means
#     f_class_means = []
#     for c in range(num_fine_classes):
#         mask = (f_labels_np == c)
#         f_class_means.append(feats_np[mask].mean(axis=0))
#     f_class_means = np.stack(f_class_means, axis=0)   # [C, D]

#     # pooled covariance
#     f_centered = []
#     c_centered = []

#     for c in range(num_fine_classes):
#         mask = (f_labels_np == c)
#         f_centered.append(feats_np[mask] - f_class_means[c])
#     f_centered = np.concatenate(f_centered, axis=0)
#     f_cov = np.cov(f_centered, rowvar=False) + EPS * np.eye(D)
#     f_inv_cov = np.linalg.pinv(f_cov)

#     if hierarchical:
#         c_class_means = []
#         for c in range(num_coarse_classes):
#             mask = (c_labels_np == c)
#             c_class_means.append(feats_np[mask].mean(axis=0))
#         c_class_means = np.stack(c_class_means, axis=0)   # [C, D]
    
#         # pooled covariance
#         for c in range(num_coarse_classes):
#             mask = (c_labels_np == c)
#             c_centered.append(feats_np[mask] - c_class_means[c])
#         c_centered = np.concatenate(c_centered, axis=0)
#         c_cov = np.cov(c_centered, rowvar=False) + EPS * np.eye(D)
#         c_inv_cov = np.linalg.pinv(c_cov)
#         stats = {
#         'f_class_means': f_class_means,
#         'c_class_means': c_class_means,
#         'f_inv_cov': f_inv_cov,
#         'c_inv_cov': c_inv_cov
#         }
#     # invert covariance
#     else:
#         stats = {
#         'f_class_means': f_class_means,
#         'f_inv_cov': f_inv_cov,
#         }
#     return stats


# # ------------------------------------------------------------
# # 3) Mahalanobis scoring (vectorized)
# # ------------------------------------------------------------
# def maha_scores(features_np, class_means, inv_cov):
#     # features_np: [M, D]
#     # class_means: [C, D]
#     # inv_cov: [D, D]
#     x = features_np[:, None, :]         # [M, 1, D]
#     mu = class_means[None, :, :]        # [1, C, D]
#     diff = x - mu                       # [M, C, D]
#     tmp = diff @ inv_cov                # [M, C, D]
#     qf = (tmp * diff).sum(axis=2)       # [M, C]
#     return -qf.min(axis=1)              # higher = more ID-like


# # ------------------------------------------------------------
# # 4) Energy scoring from logits
# # ------------------------------------------------------------
# def energy_score_from_logits(logits_np, T=1.0):
#     return -T * scipy.special.logsumexp(logits_np / T, axis=1)


# # ------------------------------------------------------------
# # 5) AUROC + FPR95 evaluation
# # ------------------------------------------------------------
def eval_detector(scores_id, scores_ood):
    y = np.concatenate([np.ones_like(scores_id), np.zeros_like(scores_ood)])
    s = np.concatenate([scores_id, scores_ood])

    auroc = roc_auc_score(y, s)

    fpr, tpr, thr = roc_curve(y, s)
    try:
        fpr95 = fpr[np.where(tpr >= 0.95)[0][0]]
    except:
        fpr95 = 1.0

    return auroc, fpr95

# #====================================================================
def get_classifier_ood_scores(model, loader, device):
    model.eval()
    scores = []

    with torch.no_grad():
        for x in loader:
            x = x[0].to(device)
            _, logits = model(x)
            probs = torch.softmax(logits, dim=1)
            # class 1 = outlier
            scores.append(probs[:, 1].cpu())
    
    return torch.cat(scores, dim=0).numpy()

# # ------------------------------------------------------------
# # ODIN scoring (Hendrycks & Gimpel 2017)
# # ------------------------------------------------------------
# def odin_scores(model, loader, device, T=1000.0, eps=0.0014):
#     model.eval()
#     scores = []

#     for batch in loader:
#         x = batch[0].to(device).float()
#         x.requires_grad = True

#         # 1) forward
#         emb, logits = model.forward_features_and_logits(x)
#         logits = logits / T
#         max_logit, _ = logits.max(dim=1)

#         # 2) gradient
#         loss = -max_logit.sum()
#         model.zero_grad()
#         loss.backward()

#         # 3) perturbation
#         x_adv = x - eps * x.grad.data.sign()
#         x_adv = torch.clamp(x_adv, 0, 1)

#         # 4) recompute
#         with torch.no_grad():
#             _, logits2 = model.forward_features_and_logits(x_adv)
#             logits2 = logits2 / T
#             energy = -T * torch.logsumexp(logits2, dim=1)

#         scores.append(energy.cpu())

#     return torch.cat(scores, dim=0).numpy()


# # ============================================================
# # Example usage (just run this block after defining loaders)
# # ============================================================
# import matplotlib.pyplot as plt
# from sklearn.metrics import roc_curve, auc

# def plot_roc_curve(method_scores, method_labels, method_name, ax=None):
#     """
#     method_scores: numpy array (higher = more inlier-like)
#     method_labels: numpy array of {1 for ID, 0 for OOD}
#     """
#     fpr, tpr, _ = roc_curve(method_labels, method_scores)
#     roc_auc = auc(fpr, tpr)

#     if ax is None:
#         fig, ax = plt.subplots(figsize=(6, 6))

#     ax.plot(fpr, tpr, label=f"{method_name} (AUC={roc_auc:.3f})")
#     ax.plot([0, 1], [0, 1], 'k--')
#     ax.set_xlabel("False Positive Rate")
#     ax.set_ylabel("True Positive Rate")
#     ax.set_title("ROC Curve")
#     ax.legend()
#     ax.grid(True)

#     return ax



# Train

In [16]:
def train(cfg):
    device = cfg["device"]
    print(f"Using device: {device}")

    # ------------------- Build datasets --------------------
    ttr = get_transforms(cfg["inlier_dataset"], "train")
    tte = get_transforms(cfg["inlier_dataset"], "test")

    # Determine dataset type and parameters
    if cfg["inlier_dataset"] == "cifar100":
        in_train = CIFAR100Hierarchy("./data", train=True, transform=ttr, download=True)
        in_test = CIFAR100Hierarchy("./data", train=False, transform=tte, download=True) # Renamed to in_test
        is_hierarchical = True
        # anchor, fine_positive, fine-negative (coarse positive), coarse_negative , outlier , noise
        N_levels_uhcl = 6 #5
        depth, embed_dim = 18, cfg["embed_dim"]
    else:
        in_train = CIFAR10Indexed("./data", train=True, transform=ttr, download=True) # Use CIFAR10Indexed
        in_test = CIFAR10Indexed("./data", train=False, transform=tte, download=True) # Use CIFAR10Indexed # Renamed to in_test
        is_hierarchical = False
        # anchor, inlier-positive, inlier-negative , outlier , noise
        N_levels_uhcl = 5 #4
        depth, embed_dim = 18, cfg["embed_dim"]

    aux_loss = 0.0

        # Build classifier heads ONLY ONCE

        



    print('outlier set: ', cfg['outlier_dataset'])
    if cfg['outlier_dataset'] == "svhn":
        out_train = datasets.SVHN("./data", split='train', transform=ttr, download=True) # Use ttr for train
        out_test = datasets.SVHN("./data", split='test', transform=tte, download=True) # Renamed to out_test
    elif cfg["outlier_dataset"] == "cifar10":
        out_train = datasets.CIFAR10("./data", train=True, transform=ttr, download=True) # Use ttr for train
        out_test = datasets.CIFAR10("./data", train=False, transform=tte, download=True) # Renamed to out_test
    elif cfg["outlier_dataset"] == "cifar100":
        out_train = CIFAR100Hierarchy("./data", train=True, transform=ttr, download=True) # Use ttr for train
        out_test = CIFAR100Hierarchy("./data", train=False, transform=tte, download=True) # Renamed to out_test
        # inlier_coarse_labels = get_random_coarse_split(full_train, split_ratio=0.5, seed=42)
        # in_train, out_train = split_dataset_by_labels(full_train, inlier_coarse_labels)
        # in_test, out_test = split_dataset_by_labels(full_test, inlier_coarse_labels)
        print('here we are')
    # Define in_channels based on the inlier dataset
    in_channels = 1 if cfg["inlier_dataset"] in ["mnist", "fashionmnist", "emnist"] else 3

    # Use the sizes of the training datasets for the noise datasets
    noise_train = RandomNoiseDataset(len(in_train), (in_channels, IMG_SIZE.get(cfg["inlier_dataset"], 32), IMG_SIZE.get(cfg["inlier_dataset"], 32)))
    noise_test = RandomNoiseDataset(len(in_test), (in_channels, IMG_SIZE.get(cfg["inlier_dataset"], 32), IMG_SIZE.get(cfg["inlier_dataset"], 32))) # Use len(in_test) for noise_test


    # Initialize model with a 3-class classifier head for anomaly/noise (inlier, outlier, noise)
    print(f"Initializing SmallResNet with pretrained={cfg["pretrained"]}") # Add this print statement
    model = SmallResNet(embed_dim=embed_dim, in_channels=in_channels,pretrained = cfg["pretrained"] ,depth=depth, num_classes=3).to(device)
    if cfg["inlier_dataset"] == "cifar100":
        model.coarse_head = nn.Linear(embed_dim, 20).to(device)
        model.fine_head   = nn.Linear(embed_dim, 100).to(device)
    else:
        model.inlier_head = nn.Linear(embed_dim, 10).to(device)
    # Load checkpoint if specified
    if cfg["resume_ckpt_path"] and os.path.exists(cfg["resume_ckpt_path"]):
        print(f"Loading checkpoint from {cfg['resume_ckpt_path']}")
        model.load_state_dict(torch.load(cfg["resume_ckpt_path"], map_location=device))
        print("Checkpoint loaded successfully.")
    elif cfg["resume_ckpt_path"]:
        print(f"Warning: Checkpoint path '{cfg['resume_ckpt_path']}' not found. Starting training from scratch.")


    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    # The main DataLoader should use the inlier training dataset for iterating through batches
    # Batch sampling for the hierarchy/outlier/noise is handled within sample_hierarchy_batch
    main_loader = DataLoader(in_train, batch_size=cfg["batch_size"], shuffle=True, num_workers=cfg["num_workers"])

    # ------------------- Training loop ---------------------
    for ep in range(cfg["epochs"]):
        model.train()
        epoch_loss = 0.0
        batch_count = 0

        pbar = tqdm(main_loader, desc=f"Epoch {ep+1}/{cfg['epochs']}")
        
        for _, _, *rest in pbar: 
            # 1. Sample the batch
            X, y_fine, y_coarse = sample_hierarchy_batch(
                in_train, out_train, noise_train,
                batch_size=cfg["batch_size"],
                device=device,
                N = N_levels_uhcl,
                is_hierarchical=is_hierarchical
            )
            
            if X.size(0) == 0:
                print("Warning: Sampled an empty batch. Skipping.")
                continue

            # 2. Forward Pass
            B, N_levels, C, H, W = X.shape
            X = X.view(B*N_levels, C, H, W) 
            Z_all , logits = model(X)
            # logits_all = model(X, return_embeddings=False)
            Z_all = Z_all.view(B, N_levels, -1)
            logits = logits.view(B, N_levels, -1)


            # 3. UHCL Loss
            # uhcl_loss = uhcl(Z_all, N = N_levels_uhcl, alpha_base= cfg['alpha_base'], beta_base = cfg['beta_base'], beta_decay=cfg['beta_decay'],lambda_pos=cfg['lambda_pos'], device=device)
            # uhcl_loss = uhcl(logits, N = N_levels_uhcl, alpha= cfg['alpha'], alpha_decay = cfg['alpha_decay'], device=device)
            

            # hyp_embs = Z_all   # but actually this is the Poincaré embedding now
            uhcl_loss = uhcl(Z_all, N = N_levels_uhcl, alpha= cfg['alpha'], alpha_decay = cfg['alpha_decay'], device=device)

            # uhcl_loss = uhcl(Z_all, N = N_levels_uhcl, alpha= cfg['alpha'], device=device)


            # 4. Auxiliary Cross-Entropy Loss (FIXED)
            if cfg["inlier_dataset"] == "cifar100":
                # For CIFAR100, we use the first 4 columns (Anchor, Pos, Neg-Fine, Neg-Coarse)
                # We must flatten both embeddings and labels to match
                inlier_embs = logits[:, :4, :].reshape(-1, embed_dim)
                
                # Use y_fine and y_coarse returned by the sampler, not X_indices
                # y_fine in sampler has 4 cols for CIFAR100 [fine0, fine0, fine2, fine3]
                fine_labels = y_fine[:, :4].reshape(-1) 
                coarse_labels = y_coarse[:, :4].reshape(-1)
            
                fine_logits = model.fine_head(inlier_embs)
                coarse_logits = model.coarse_head(inlier_embs)

                aux_loss = F.cross_entropy(fine_logits, fine_labels) + \
                           F.cross_entropy(coarse_logits, coarse_labels)
            
            else:
                # For CIFAR10, we use the first 3 columns (Anchor, Pos, Neg)
                # Z_all shape: [Batch_Size, N, Embed_Dim] -> Take first 3 columns
                inlier_embs = logits[:, :3, :].reshape(-1, embed_dim) # Shape: [Batch*3, Embed_Dim]
                
                # --- THE FIX IS HERE ---
                # Old (Error): class_labels = torch.tensor(in_train.targets, ...) -> Size 50000
                # New (Correct): Use y_fine from the batch sampler -> Size Batch*3
                class_labels = y_fine[:, :3].reshape(-1) 
                
                logits = model.inlier_head(inlier_embs) # Shape: [Batch*3, 10]
                
                aux_loss = F.cross_entropy(logits, class_labels)

            total_loss = uhcl_loss + (cfg["aux_weight"] * aux_loss)

            # 5. Backpropagation
            opt.zero_grad()
            total_loss.backward()
            opt.step()

            batch_loss = uhcl_loss.item()
            epoch_loss += batch_loss
            batch_count += 1
            
            pbar.set_postfix({"UHCL": f"{batch_loss:.4f}", "Aux": f"{aux_loss.item():.4f}"})
        if ep % 10 == 0:  # Checkpoint every 10 epochs
            checkpoint_path = os.path.join(config["ckpt_dir"], f"{config['experiment_name']}_ep{ep}.pth")
            os.makedirs(config["ckpt_dir"], exist_ok=True)  
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Model checkpoint saved to {checkpoint_path}")
        
    print("\n============ Running Mahalanobis / Energy / ODIN scoring ============")
    checkpoint_path = os.path.join(config["ckpt_dir"], f"{config['experiment_name']}_last.pth")
    torch.save(model.state_dict(), checkpoint_path)
    print(f"Model checkpoint saved to {checkpoint_path}")

    # ------------------- Train classification head --------------------
    train_clf(model, in_train, out_train, noise_train, device, epochs = 10) # train the classifier head with learned embeddings of the main model and actuall datalabels


    

    # ------------------- Evaluate and print summary --------------------
    print("\n--- Evaluation Summary ---")
    (inlier_fine_acc_train, inlier_fine_precision_train, inlier_fine_recall_train, inlier_fine_f1_train,
     inlier_coarse_acc_train, inlier_coarse_precision_train, inlier_coarse_recall_train, inlier_coarse_f1_train,
     inlier_fine_acc_test, inlier_fine_precision_test, inlier_fine_recall_test, inlier_fine_f1_test,
     inlier_coarse_acc_test, inlier_coarse_precision_test, inlier_coarse_recall_test, inlier_coarse_f1_test,
     auroc_out_clf, precision_out_clf, recall_out_clf, f1_out_clf,
     auroc_noise_clf, precision_noise_clf, recall_noise_clf, f1_noise_clf) = evaluate_summary(model, in_train, in_test, out_test, noise_test, device, cfg["experiment_name"])
    # ---- Build loaders ----

    train_loader_inlier = DataLoader(in_train, batch_size=256, shuffle=False)
    test_loader_inlier  = DataLoader(in_test,  batch_size=256, shuffle=False)
    test_loader_outlier = DataLoader(out_test, batch_size=256, shuffle=False)
    
    # # ---- Compute stats for Mahalanobis ----
    # if not is_hierarchical:
    #     stats = compute_maha_stats(model, train_loader_inlier, False)
    # else:
    #     stats = compute_maha_stats(model, train_loader_inlier, True)

    
    # # ---- Extract features/logits for ID+OOD sets ----
    # if  not is_hierarchical:
    #     id_feats,  id_logits,  _ = extract_features_and_logits(model, test_loader_inlier, False)
    #     ood_feats, ood_logits, _ = extract_features_and_logits(model, test_loader_outlier, False)
        

    # else:
    #     id_feats,  id_logits,  _, _ = extract_features_and_logits(model, test_loader_inlier, True)
    #     ood_feats, ood_logits, _, _ = extract_features_and_logits(model, test_loader_outlier, False)
        
    # id_feats_np  = id_feats.numpy()
    # ood_feats_np = ood_feats.numpy()
    # id_logits_np  = id_logits.numpy()
    # ood_logits_np = ood_logits.numpy()
    
    # # ---- Mahalanobis ----
    # maha_f_id  = maha_scores(id_feats_np,  stats['f_class_means'], stats['f_inv_cov'])
    # maha_f_ood = maha_scores(ood_feats_np, stats['f_class_means'], stats['f_inv_cov'])
    # maha_f_auroc, maha_fpr95f = eval_detector(maha_f_id, maha_f_ood)
    # if  is_hierarchical:
    #     maha_c_id  = maha_scores(id_feats_np,  stats['c_class_means'], stats['c_inv_cov'])
    #     maha_c_ood = maha_scores(ood_feats_np, stats['c_class_means'], stats['c_inv_cov'])
    #     maha_c_auroc, maha_fpr95c = eval_detector(maha_c_id, maha_c_ood)
    # # ---- Energy ----
    # energy_id  = energy_score_from_logits(id_logits_np,  T=1.0)
    # energy_ood = energy_score_from_logits(ood_logits_np, T=1.0)
    # energy_auroc, energy_fpr95 = eval_detector(energy_id, energy_ood)
    
    # # ---- ODIN ----
    # odin_id  = odin_scores(model, test_loader_inlier,  device, T=1000.0, eps=0.0014)
    # odin_ood = odin_scores(model, test_loader_outlier, device, T=1000.0, eps=0.0014)
    # odin_auroc, odin_fpr95 = eval_detector(odin_id, odin_ood)
    
    # ---- Print results ----
    # if not is_hierarchical:
    #     print("============== OOD Detector Results ==============")
    #     print(f"Mahalanobis: AUROC={maha_f_auroc:.4f}, FPR95={maha_fpr95f:.4f}")
    #     print(f"Energy:      AUROC={energy_auroc:.4f}, FPR95={energy_fpr95:.4f}")
    #     print(f"ODIN:        AUROC={odin_auroc:.4f}, FPR95={odin_fpr95:.4f}")
    #     print("==================================================\n")
    # else:
    #     print("============== OOD Detector Results ==============")
    #     print(f"Mahalanobis: AUROC (fine)={maha_f_auroc:.4f}, FPR95={maha_fpr95f:.4f}")
    #     print(f"Mahalanobis: AUROC (coarse)={maha_c_auroc:.4f}, FPR95={maha_fpr95c:.4f}")
    #     print(f"Energy:      AUROC={energy_auroc:.4f}, FPR95={energy_fpr95:.4f}")
    #     print(f"ODIN:        AUROC={odin_auroc:.4f}, FPR95={odin_fpr95:.4f}")
    #     print("==================================================\n")

    # ----- Build labels that match the score lengths -----
    
   
    # id_labels = np.ones(len(maha_f_id))
    # ood_labels = np.zeros(len(maha_f_ood))
    # ----- Build unified score arrays -----
    # maha_scores_all = np.concatenate([maha_id, maha_ood])
    # energy_scores_all = np.concatenate([energy_id, energy_ood])
    # odin_scores_all = np.concatenate([odin_id, odin_ood])
    clf_id  = get_classifier_ood_scores(model, test_loader_inlier,  device)
    clf_ood = get_classifier_ood_scores(model, test_loader_outlier, device)
    clf_auroc, clf_fpr95 = eval_detector(clf_id, clf_ood)

    # maha_scores_all   = np.concatenate([maha_c_id, maha_c_ood])
    # energy_scores_all = np.concatenate([energy_id, energy_ood])
    # odin_scores_all   = np.concatenate([odin_id, odin_ood])
    # clf_scores_all    = np.concatenate([clf_id, clf_ood])
    # labels = np.concatenate([
    #     np.ones_like(maha_f_id),        # ID = 1
    #     np.zeros_like(maha_f_ood)       # OOD = 0
    # ])
    # ax = plot_roc_curve(maha_scores_all, labels, "Mahalanobis")
    # plot_roc_curve(energy_scores_all, labels, "Energy", ax=ax)
    # plot_roc_curve(odin_scores_all,   labels, "ODIN", ax=ax)
    # plot_roc_curve(clf_scores_all,    labels, "Classifier", ax=ax)

    # ----- Plot ROC curves -----
    # ax = plot_roc_curve(maha_scores_all, labels, "Mahalanobis")
    # plot_roc_curve(energy_scores_all, labels, "Energy", ax=ax)
    # plot_roc_curve(odin_scores_all, labels, "ODIN", ax=ax)
    # plot_roc_curve(clf_scores_all * -1,  labels, "Classifier", ax=ax)
    # plt.show()


    print(f"Final Inlier Fine Classification Accuracy (KNN) (Train): {inlier_fine_acc_train:.4f}")
    print(f"Final Inlier Fine Classification Precision (KNN) (Train): {inlier_fine_precision_train:.4f}")
    print(f"Final Inlier Fine Classification Recall (KNN) (Train): {inlier_fine_recall_train:.4f}")
    print(f"Final Inlier Fine Classification F1-score (KNN) (Train): {inlier_fine_f1_train:.4f}")
    if is_hierarchical:
        print(f"Final Inlier Coarse Classification Accuracy (KNN) (Train): {inlier_coarse_acc_train:.4f}")
        print(f"Final Inlier Coarse Classification Precision (KNN) (Train): {inlier_coarse_precision_train:.4f}")
        print(f"Final Inlier Coarse Classification Recall (KNN) (Train): {inlier_coarse_recall_train:.4f}")
        print(f"Final Inlier Coarse Classification F1-score (KNN) (Train): {inlier_coarse_f1_train:.4f}")

    print(f"Final Inlier Fine Classification Accuracy (KNN) (Test): {inlier_fine_acc_test:.4f}")
    print(f"Final Inlier Fine Classification Precision (KNN) (Test): {inlier_fine_precision_test:.4f}")
    print(f"Final Inlier Fine Classification Recall (KNN) (Test): {inlier_fine_recall_test:.4f}")
    print(f"Final Inlier Fine Classification F1-score (KNN) (Test): {inlier_fine_f1_test:.4f}")
    if is_hierarchical:
        print(f"Final Inlier Coarse Classification Accuracy (KNN) (Test): {inlier_coarse_acc_test:.4f}")
        print(f"Final Inlier Coarse Classification Precision (KNN) (Test): {inlier_coarse_precision_test:.4f}")
        print(f"Final Inlier Coarse Classification Recall (KNN) (Test): {inlier_coarse_recall_test:.4f}")
        print(f"Final Inlier Coarse Classification F1-score (KNN) (Test): {inlier_coarse_f1_test:.4f}")

    print(f"Final OOD AUROC (Classifier): {auroc_out_clf:.4f}")
    print(f"Final OOD Precision (Classifier): {precision_out_clf:.4f}")
    print(f"Final OOD Recall (Classifier): {recall_out_clf:.4f}")
    print(f"Final OOD F1-score (Classifier): {f1_out_clf:.4f}")
    print(f"Final Noise AUROC (Classifier): {auroc_noise_clf:.4f}")
    print(f"Final Noise Precision (Classifier): {precision_noise_clf:.4f}")
    print(f"Final Noise Recall (Classifier): {recall_noise_clf:.4f}")
    print(f"Final Noise F1-score (Classifier): {f1_noise_clf:.4f}")


    # ------------------- Return trained model and datasets --------------------
    return model, in_train, in_test, out_test, noise_test

#  RUN

In [17]:
if __name__=="__main__":
    print("Starting UHCL updated runner")
    model, in_train, in_test, out_test, noise_test = train(config)

Starting UHCL updated runner
Using device: cuda
outlier set:  cifar100
here we are
Initializing SmallResNet with pretrained=True


Epoch 1/100: 100%|█████████████████████████| 500/500 [03:12<00:00,  2.60it/s, UHCL=0.4715, Aux=8.6778]


Model checkpoint saved to ./content/checkpoints/2222_ep0.pth


Epoch 11/100: 100%|███████████████████████| 500/500 [03:15<00:00,  2.55it/s, UHCL=0.4252, Aux=13.6668]


Model checkpoint saved to ./content/checkpoints/2222_ep10.pth


...

Model checkpoint saved to ./content/checkpoints/2222_ep90.pth


Epoch 100/100: 100%|██████████████████████| 500/500 [03:16<00:00,  2.55it/s, UHCL=0.3968, Aux=26.6447]



============ Running Mahalanobis / Energy / ODIN scoring ============
Model checkpoint saved to ./content/checkpoints/2222_last.pth

--- Training 3-Class Classification Head (Linear Probe) ---
Extracting features from frozen backbone...


Class 2: 100%|██████████████████████████████████████████████████████| 196/196 [00:02<00:00, 91.63it/s]


Head Epoch 1/10 | Loss: 2.4913 | Acc: 40.65%
Head Epoch 2/10 | Loss: 0.6550 | Acc: 66.62%
Head Epoch 3/10 | Loss: 0.5210 | Acc: 66.54%
Head Epoch 4/10 | Loss: 0.4958 | Acc: 66.49%
Head Epoch 5/10 | Loss: 0.4842 | Acc: 66.72%
Head Epoch 6/10 | Loss: 0.4777 | Acc: 66.69%
Head Epoch 7/10 | Loss: 0.4736 | Acc: 66.71%
Head Epoch 8/10 | Loss: 0.4708 | Acc: 66.62%
Head Epoch 9/10 | Loss: 0.4689 | Acc: 66.56%
Head Epoch 10/10 | Loss: 0.4675 | Acc: 66.69%
Classification Head Training Complete.

--- Evaluation Summary ---
Evaluating model: 2222
Final Inlier Fine Classification Accuracy (KNN) (Train): 0.0547
Final Inlier Fine Classification Precision (KNN) (Train): 0.0554
Final Inlier Fine Classification Recall (KNN) (Train): 0.0547
Final Inlier Fine Classification F1-score (KNN) (Train): 0.0492
Final Inlier Coarse Classification Accuracy (KNN) (Train): 0.1597
Final Inlier Coarse Classification Precision (KNN) (Train): 0.1552
Final Inlier Coarse Classification Recall (KNN) (Train): 0.1597
Final I

In [18]:
# Save the trained model
